# ۴. تخمین فشار در ویزیت بازگشت

این نوت‌بوک به تخمین فشار می‌رسد، ولی خط لولهٔ سیگنال خودمان را وارد مدل نمی‌کند. آشکارساز قله روی راه‌رفتن قابل اتکا نبود و ویژگی HRV نوری را هم از روی موج نساختیم.

ورودی، ستون‌های دلتای `features.tsv` منتشرشده است. این بازتولید بخش استخراج ویژگی مقاله نیست. مقایسهٔ مدل، تقسیم شرکت‌کننده، و دو خط پایه را می‌سنجد.

عدد اصلی هر خانه، میانگین بایاس و میانگین انحراف معیار روی ۱۰۰ نمونهٔ ۹۰درصدی از شرکت‌کننده‌هاست. خطا برابر پیش‌بینی منهای مرجع است. آلفا برابر ۱ روی ویژگی استانداردشده در هر فولد است؛ مقاله آلفا را نداده و این عدد تنظیم نویسنده نیست.

جدول مقاله برای تونومتری سمعی در کل جمعیت، SBP را حدود `0.32±9.75` و DBP را حدود `0.54±7.65` گزارش کرده است. این دو عدد از روی تصویر جدول خوانده شده‌اند. انتظار برابری با آن عددها را نداشته باشید: خط پایهٔ فایل «میانگین کالیبراسیون» است، آلفا مجهول است، و گروه نوری و ساعت هوشمند ستون HRV نوری را ندارند.


In [6]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('خارج از کولب؛ اتصال درایو رد شد.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import base64
from pathlib import Path
root = Path('/content/aurorabp_reproduction')
(root / 'src').mkdir(parents=True, exist_ok=True)
(root / 'src' / '__init__.py').write_text('__version__ = "0.1.0"\n', encoding='utf-8')
(root / 'src' / 'bp_protocol.py').write_bytes(base64.b64decode("..."))
print(root / 'src' / 'bp_protocol.py')


/content/aurorabp_reproduction/src/bp_protocol.py


In [8]:
import sys
from pathlib import Path
sys.path.insert(0, '/content/aurorabp_reproduction')
import numpy as np
import pandas as pd
from src.bp_protocol import run_return_visit

candidates = [
    Path('/content/aurora_waveforms'),
    Path('/content/drive/MyDrive/aurora_dataset'),
]
data_dir = next(path for path in candidates if (path / 'features.tsv').is_file())
print('داده:', data_dir)
features = pd.read_csv(data_dir / 'features.tsv', sep='\t')
participants = pd.read_csv(data_dir / 'participants.tsv', sep='\t')
gap_sbp = (features['sbp'] - features['baseline_sbp'] - features['delta_sbp']).abs()
gap_dbp = (features['dbp'] - features['baseline_dbp'] - features['delta_dbp']).abs()
print(f'بیشینه اختلاف delta_sbp با sbp-baseline: {gap_sbp.max():.6f}')
print(f'بیشینه اختلاف delta_dbp با dbp-baseline: {gap_dbp.max():.6f}')
if gap_sbp.max() > 0.05 or gap_dbp.max() > 0.05:
    raise RuntimeError('تعریف دلتای فشار با ستون‌های فایل یکی نیست.')


داده: /content/drive/MyDrive/aurora_dataset
بیشینه اختلاف delta_sbp با sbp-baseline: 0.000000
بیشینه اختلاف delta_dbp با dbp-baseline: 0.000000


In [9]:
output = Path('/content/aurorabp_outputs/return_visit_bp.json')
output.parent.mkdir(parents=True, exist_ok=True)
results = run_return_visit(features, participants, output_path=str(output))
print(output)
results


provenance: released participants {'auscultatory': 673, 'oscillometric': 548, 'total': 1221}
provenance: analyzed (in_feature_table) {'auscultatory': 642, 'oscillometric': 483, 'total': 1125}
provenance: distinct pids inside features.tsv {'auscultatory': 642, 'oscillometric': 483, 'total': 1125}
provenance: auscultatory analyzed 642 versus figure 642
provenance: oscillometric analyzed 483 versus figure 483
provenance: participants outside the analyzed box 96
provenance: model input 1125 participants, 40256 feature rows
sensor models include baseline_sbp and baseline_dbp; static baseline unchanged
return visit in-lab only: phase=return, 1123 participants, 4767 rows; ambulatory excluded
auscultatory all ecg delta_sbp: ridge 0.00±10.36  zero 1.58±10.49  static -0.00±10.37  n=3154
auscultatory all ecg delta_dbp: ridge -0.00±7.85  zero 1.70±8.19  static -0.00±7.90  n=3153
auscultatory all optical delta_sbp: ridge -0.00±10.34  zero 1.67±10.53  static -0.00±10.42  n=2775
auscultatory all opti

,protocol,population,row_set,model,target,missing_table_features,quality_column,exercise_rows_removed,reliability,n_rows,n_participants,lopo_bias,lopo_sd,boot_bias,boot_sd,boot_bias_p2_5,boot_bias_p97_5,boot_replicates_used
0,auscultatory,all,ecg,ridge,delta_sbp,,,0,official features plus calibration SBP/DBP; al...,3154,635,0.000930,10.379247,0.001340,10.360576,-0.002720,0.004551,100
1,auscultatory,all,ecg,zero,delta_sbp,,,0,official features plus calibration SBP/DBP; al...,3154,635,1.563729,10.514299,1.577333,10.490117,1.325378,1.803286,100
2,auscultatory,all,ecg,static,delta_sbp,,,0,official features plus calibration SBP/DBP; al...,3154,635,-0.002008,10.391086,-0.001925,10.369187,-0.004017,0.001271,100
3,auscultatory,all,ecg,ridge,delta_dbp,,,0,official features plus calibration SBP/DBP; al...,3153,635,-0.000183,7.839081,-0.000125,7.846168,-0.002453,0.002662,100
4,auscultatory,all,ecg,zero,delta_dbp,,,0,official features plus calibration SBP/DBP; al...,3153,635,1.690295,8.195246,1.700177,8.193978,1.489384,1.905367,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,oscillometric,normotensive,tonometric,zero,delta_sbp,,quality_pressure,0,official features plus calibration SBP/DBP; al...,398,206,-7.806533,9.601060,-7.815573,9.599076,-8.146156,-7.397572,100
116,oscillometric,normotensive,tonometric,static,delta_sbp,,quality_pressure,0,official features plus calibration SBP/DBP; al...,398,206,0.016077,9.345315,0.014898,9.355497,-0.036128,0.040985,100
117,oscillometric,normotensive,tonometric,ridge,delta_dbp,,quality_pressure,0,official features plus calibration SBP/DBP; al...,398,206,0.023397,7.338540,0.022883,7.379770,-0.022455,0.078933,100
118,oscillometric,normotensive,tonometric,zero,delta_dbp,,quality_pressure,0,official features plus calibration SBP/DBP; al...,398,206,-8.924623,7.498403,-8.940639,7.504003,-9.252957,-8.643309,100


In [10]:
show = results.loc[results['model'].isin(['ridge', 'zero', 'static']), [
    'protocol', 'population', 'row_set', 'model', 'target', 'n_rows', 'n_participants',
    'boot_bias', 'boot_sd', 'lopo_bias', 'lopo_sd', 'reliability',
]].copy()
for column in ('boot_bias', 'boot_sd', 'lopo_bias', 'lopo_sd'):
    show[column] = show[column].map(lambda value: round(float(value), 2) if pd.notna(value) else value)
headline = show.loc[(show['population'] == 'all') & (show['row_set'] == 'tonometric')]
print('تونومتری، کل جمعیت. boot_bias و boot_sd عدد گزارش‌اند. lopo فقط بررسی روی همهٔ شرکت‌کننده‌هاست و آمار مقاله نیست.')
headline


تونومتری، کل جمعیت. boot_bias و boot_sd عدد گزارش‌اند. lopo فقط بررسی روی همهٔ شرکت‌کننده‌هاست و آمار مقاله نیست.


,protocol,population,row_set,model,target,n_rows,n_participants,boot_bias,boot_sd,lopo_bias,lopo_sd,reliability
18,auscultatory,all,tonometric,ridge,delta_sbp,2796,632,-0.00,9.89,-0.00,9.86,official features plus calibration SBP/DBP; al...
19,auscultatory,all,tonometric,zero,delta_sbp,2796,632,1.49,10.40,1.49,10.38,official features plus calibration SBP/DBP; al...
20,auscultatory,all,tonometric,static,delta_sbp,2796,632,-0.00,10.27,-0.00,10.25,official features plus calibration SBP/DBP; al...
21,auscultatory,all,tonometric,ridge,delta_dbp,2795,632,0.01,7.71,0.00,7.69,official features plus calibration SBP/DBP; al...
22,auscultatory,all,tonometric,zero,delta_dbp,2795,632,1.69,8.17,1.68,8.17,official features plus calibration SBP/DBP; al...
23,auscultatory,all,tonometric,static,delta_dbp,2795,632,0.00,7.84,0.00,7.83,official features plus calibration SBP/DBP; al...
66,oscillometric,all,tonometric,ridge,delta_sbp,577,310,-0.02,10.65,-0.01,10.61,official features plus calibration SBP/DBP; al...
67,oscillometric,all,tonometric,zero,delta_sbp,577,310,-5.69,11.22,-5.70,11.23,official features plus calibration SBP/DBP; al...
68,oscillometric,all,tonometric,static,delta_sbp,577,310,0.01,10.65,0.01,10.64,official features plus calibration SBP/DBP; al...
69,oscillometric,all,tonometric,ridge,delta_dbp,577,310,0.01,7.64,0.01,7.62,official features plus calibration SBP/DBP; al...
